# 💹 Financial Knowledge Hybrid Search Engine

A hybrid retrieval system over **2,196 real financial document chunks** sourced from Wikipedia and ArXiv.

| Component | Detail |
|---|---|
| Data Sources | Wikipedia (20 articles) + ArXiv (10 papers) |
| Dense Embeddings | `all-MiniLM-L6-v2` (HuggingFace, 384-dim) |
| Sparse Encoding | BM25 (`pinecone-text`) |
| Vector Database | Pinecone Serverless (AWS us-east-1, dotproduct) |
| Retriever | `PineconeHybridSearchRetriever` (LangChain) |

> **Benchmark Results:** Hybrid fusion (alpha=0.5) achieved **MRR@5 of 0.73** and **86.7% Hit Rate@5**,
> outperforming pure BM25 by **+14.2%** and pure semantic by **+18.6%** MRR on 15 hard financial queries.

## 1. Install Dependencies

In [1]:
!pip install --quiet --upgrade \
    pinecone pinecone-text \
    langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface \
    sentence-transformers huggingface_hub \
    python-dotenv wikipedia arxiv streamlit


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Load Environment Variables

In [2]:
import os, time, json, statistics
from dotenv import load_dotenv

load_dotenv()
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
HF_TOKEN         = os.getenv("HF_TOKEN")
os.environ["HF_TOKEN"] = HF_TOKEN

assert PINECONE_API_KEY, "PINECONE_API_KEY missing from .env"
assert HF_TOKEN,         "HF_TOKEN missing from .env"
print("✅ Environment variables loaded.")

✅ Environment variables loaded.


## 3. Initialize Pinecone — Create Fresh Index

In [3]:
from pinecone import Pinecone, ServerlessSpec

INDEX_NAME = "financial-hybrid-search"
DIMENSION  = 384
METRIC     = "dotproduct"

pc = Pinecone(api_key=PINECONE_API_KEY)
existing = [idx.name for idx in pc.list_indexes()]

if INDEX_NAME in existing:
    print(f"Deleting old index...")
    pc.delete_index(INDEX_NAME)
    time.sleep(5)

print(f"Creating index '{INDEX_NAME}'...")
pc.create_index(
    name=INDEX_NAME, dimension=DIMENSION, metric=METRIC,
    spec=ServerlessSpec(cloud="aws", region="us-east-1"),
)
while not pc.describe_index(INDEX_NAME).status["ready"]:
    time.sleep(3)

index = pc.Index(INDEX_NAME)
print(f"✅ Index ready.")

Deleting old index...
Creating index 'financial-hybrid-search'...
✅ Index ready.


## 4. Fetch Wikipedia Articles — 20 Financial Topics

In [4]:
import wikipedia

WIKI_TOPICS = [
    "Modern portfolio theory", "Capital asset pricing model", "Value at risk",
    "Systematic risk", "Sharpe ratio", "Financial derivatives", "Hedge fund",
    "Exchange-traded fund", "Bond (finance)", "Options (finance)",
    "Fundamental analysis", "Technical analysis", "Quantitative analysis (finance)",
    "Financial ratio", "Discounted cash flow", "Stock market",
    "Foreign exchange market", "Financial crisis", "Algorithmic trading",
    "Behavioral finance",
]

wiki_docs = []
for topic in WIKI_TOPICS:
    try:
        page = wikipedia.page(topic, auto_suggest=False)
        wiki_docs.append({"text": page.content, "source": f"Wikipedia: {topic}"})
        print(f"  ✅ {topic} ({len(page.content):,} chars)")
    except Exception as e:
        print(f"  ⚠️  Skipped: {topic} — {e}")

print(f"\n✅ Fetched {len(wiki_docs)} Wikipedia articles")

  ✅ Modern portfolio theory (51,506 chars)
  ✅ Capital asset pricing model (20,852 chars)
  ✅ Value at risk (37,817 chars)
  ✅ Systematic risk (13,257 chars)
  ✅ Sharpe ratio (17,413 chars)
  ✅ Financial derivatives (59,596 chars)
  ✅ Hedge fund (69,772 chars)
  ✅ Exchange-traded fund (33,928 chars)
  ✅ Bond (finance) (35,196 chars)
  ✅ Options (finance) (39,763 chars)
  ✅ Fundamental analysis (8,352 chars)
  ✅ Technical analysis (39,049 chars)
  ✅ Quantitative analysis (finance) (23,778 chars)
  ✅ Financial ratio (7,666 chars)
  ✅ Discounted cash flow (21,409 chars)
  ✅ Stock market (40,521 chars)
  ✅ Foreign exchange market (41,858 chars)
  ✅ Financial crisis (43,360 chars)
  ✅ Algorithmic trading (51,279 chars)
  ✅ Behavioral finance (16,620 chars)

✅ Fetched 20 Wikipedia articles


## 5. Fetch ArXiv Papers — Finance + LLM Research

In [5]:
import arxiv

ARXIV_QUERIES = [
    "large language models financial analysis",
    "NLP sentiment analysis stock market prediction",
    "transformer models portfolio optimization",
    "retrieval augmented generation finance",
    "deep learning algorithmic trading",
]

arxiv_docs = []
client = arxiv.Client()
for query in ARXIV_QUERIES:
    search = arxiv.Search(query=query, max_results=2, sort_by=arxiv.SortCriterion.Relevance)
    for paper in client.results(search):
        text = f"Title: {paper.title}\n\nAbstract: {paper.summary}\n\nAuthors: {', '.join(a.name for a in paper.authors)}"
        arxiv_docs.append({"text": text, "source": f"ArXiv: {paper.title}"})
        print(f"  ✅ {paper.title[:70]}...")

print(f"\n✅ Fetched {len(arxiv_docs)} ArXiv papers")

  ✅ Construction of a Japanese Financial Benchmark for Large Language Mode...
  ✅ Beyond Classification: Financial Reasoning in State-of-the-Art Languag...
  ✅ JU_KS@SAIL_CodeMixed-2017: Sentiment Analysis for Indian Code Mixed So...
  ✅ Innovative Sentiment Analysis and Prediction of Stock Price Using FinB...
  ✅ Optimal Portfolio with Power Utility of Absolute and Relative Wealth...
  ✅ Portfolio Optimization...
  ✅ AR-RAG: Autoregressive Retrieval Augmentation for Image Generation...
  ✅ Intelligent Interaction Strategies for Context-Aware Cognitive Augment...
  ✅ Learn to Accumulate Evidence from All Training Samples: Theory and Pra...
  ✅ The Modern Mathematics of Deep Learning...

✅ Fetched 10 ArXiv papers


## 6. Chunk Documents — LangChain Text Splitter

In [6]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500, chunk_overlap=50,
    separators=["\n\n", "\n", ". ", " "],
)

all_chunks, all_sources = [], []
for doc in wiki_docs + arxiv_docs:
    chunks = splitter.split_text(doc["text"])
    all_chunks.extend(chunks)
    all_sources.extend([doc["source"]] * len(chunks))

print(f"✅ Total chunks: {len(all_chunks)}")
print(f"   Wikipedia: {sum(1 for s in all_sources if 'Wikipedia' in s)} | ArXiv: {sum(1 for s in all_sources if 'ArXiv' in s)}")

✅ Total chunks: 2196
   Wikipedia: 2144 | ArXiv: 52


## 7. Load Embedding Model + Fit BM25

In [7]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name="all-MiniLM-L6-v2",
    model_kwargs={"device": "cpu"},
    encode_kwargs={"normalize_embeddings": True},
)
print(f"✅ Embedding model loaded. Dimension: {len(embeddings.embed_query('test'))}")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Embedding model loaded. Dimension: 384


In [8]:
from pinecone_text.sparse import BM25Encoder

print(f"Fitting BM25 on {len(all_chunks)} chunks...")
bm25_encoder = BM25Encoder()
bm25_encoder.fit(all_chunks)
bm25_encoder.dump("bm25_financial.json")
bm25_encoder = BM25Encoder().load("bm25_financial.json")
print("✅ BM25 fitted and saved.")

Fitting BM25 on 2196 chunks...


  0%|          | 0/2196 [00:00<?, ?it/s]

✅ BM25 fitted and saved.


## 8. Build Retriever + Upload to Pinecone

⚠️ This cell takes **2–5 minutes** to complete.

In [9]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

retriever = PineconeHybridSearchRetriever(
    embeddings=embeddings, sparse_encoder=bm25_encoder,
    index=index, top_k=5, alpha=0.5,
)

# Filter chunks that produce empty BM25 sparse vectors
def is_valid_chunk(text):
    return len(bm25_encoder.encode_documents([text])[0].get("indices", [])) > 0

valid_chunks = [c for c in all_chunks if is_valid_chunk(c)]
print(f"Valid chunks: {len(valid_chunks)} / {len(all_chunks)}")

BATCH_SIZE = 50
for i in range(0, len(valid_chunks), BATCH_SIZE):
    retriever.add_texts(valid_chunks[i:i+BATCH_SIZE])
    print(f"  Uploaded {min(i+BATCH_SIZE, len(valid_chunks))}/{len(valid_chunks)}")

time.sleep(3)
stats = index.describe_index_stats()
print(f"\n✅ Done! Vectors in Pinecone: {stats['total_vector_count']}")

Valid chunks: 2195 / 2196


  0%|          | 0/2 [00:00<?, ?it/s]

  Uploaded 50/2195


  0%|          | 0/2 [00:00<?, ?it/s]

  Uploaded 100/2195


...

  Uploaded 2100/2195


  0%|          | 0/2 [00:00<?, ?it/s]

  Uploaded 2150/2195


  0%|          | 0/2 [00:00<?, ?it/s]

  Uploaded 2195/2195

✅ Done! Vectors in Pinecone: 2186


## 9. Query the Financial Search Engine

In [10]:
def search(query, alpha=0.5, top_k=5):
    retriever.alpha = alpha
    retriever.top_k = top_k
    results = retriever.invoke(query)
    print(f"\n🔎 '{query}'  (alpha={alpha})")
    print("-" * 70)
    for i, doc in enumerate(results, 1):
        print(f"[{i}] {doc.page_content[:200]}...\n")
    return results

search("What is systematic risk in portfolio management?")
search("How do hedge funds generate alpha?")
search("LLM applications in financial forecasting")
search("How is Value at Risk calculated?")


🔎 'What is systematic risk in portfolio management?'  (alpha=0.5)
----------------------------------------------------------------------
[1] Systematic risk plays an important role in portfolio allocation. Risk which cannot be eliminated through diversification commands returns in excess of the risk-free rate (while idiosyncratic risk does...

[2] === Systematic risk and specific risk ===

The total risk of an individual asset or portfolio is decomposed into two distinct components: specific risk and systematic risk....

[3] == Risk and diversification ==
The risk of a portfolio comprises systematic risk, also known as undiversifiable risk, and unsystematic risk, which is also known as idiosyncratic risk or diversifiable ...

[4] . Therefore, an investor's desired returns correspond with their desired exposure to systematic risk and corresponding asset selection. Investors can only reduce a portfolio's exposure to systematic r...

[5] Unsystematic risk can be diversified away to small

[Document(metadata={'score': 0.499659926}, page_content='Determined growth rates (of income and cash) and risk levels (to determine the discount rate) are used in various valuation models. The foremost is the discounted cash flow model, which calculates the present value of the future:'),
 Document(metadata={'score': 0.475793183}, page_content='Value at risk (VaR) is a measure of the risk of loss of investment/capital. It estimates how much a set of investments might lose (with a given probability), given normal market conditions, in a set time period such as a day. VaR is typically used by firms and regulators in the financial industry to gauge the amount of assets needed to cover possible losses.'),
 Document(metadata={'score': 0.474450976}, page_content='Tools\n"The Pricing and Trading of Interest Rate Derivatives", J H M Darbyshire, MSc.\nOnline real-time VaR calculator, Razvan Pascalau, University of Alabama\nValue-at-Risk (VaR), Simon Benninga and Zvi Wiener. (Mathematica in Educ

## 10. Alpha Tuning — Keyword vs Semantic Balance

In [11]:
query = "risk adjusted return portfolio optimization"
for alpha_val, label in [(0.0, "Pure BM25"), (0.5, "Hybrid"), (1.0, "Pure Semantic")]:
    retriever.alpha = alpha_val
    results = retriever.invoke(query)
    print(f"\nalpha={alpha_val} | {label}")
    for i, doc in enumerate(results[:3], 1):
        print(f"  [{i}] {doc.page_content[:120]}...")
retriever.alpha = 0.5


alpha=0.0 | Pure BM25
  [1] == Efficient frontier ==
The CAPM assumes that the risk-return profile of a portfolio can be optimized—an optimal portfo...
  [2] Additionally, since each additional asset introduced into a portfolio further diversifies the portfolio, the optimal por...
  [3] In finance, the Sharpe ratio (also known as the Sharpe index, the Sharpe measure, and the reward-to-variability ratio) m...

alpha=0.5 | Hybrid
  [1] == Efficient frontier ==
The CAPM assumes that the risk-return profile of a portfolio can be optimized—an optimal portfo...
  [2] Additionally, since each additional asset introduced into a portfolio further diversifies the portfolio, the optimal por...
  [3] Title: Portfolio Optimization...

alpha=1.0 | Pure Semantic
  [1] Title: Portfolio Optimization...
  [2] A rational investor should not take on any diversifiable risk, as only non-diversifiable risks are rewarded within the s...
  [3] {\displaystyle \rho <1}
  
, the portfolio standard deviation will

## 11. Hard Benchmark Evaluation

**3 query categories** designed to force real differentiation between retrieval methods:

| Category | Challenge |
|---|---|
| `semantic-hard` | Paraphrased — no keyword overlap, BM25 will fail |
| `bm25-hard` | Exact terms but ambiguous across topics, semantic needed |
| `hybrid-wins` | Needs BOTH keyword precision AND semantic understanding |

In [12]:
HARD_EVAL_SET = [
    # Semantic-Hard
    {"query": "measuring how much money you could lose on a bad day",             "must_contain": "value at risk",    "must_not": "sharpe",       "category": "semantic-hard"},
    {"query": "spreading investments to reduce exposure to any single asset",     "must_contain": "diversif",         "must_not": "derivative",   "category": "semantic-hard"},
    {"query": "using past price patterns to predict where a stock goes next",     "must_contain": "technical",        "must_not": "fundamental",  "category": "semantic-hard"},
    {"query": "investors making irrational decisions based on emotions not data", "must_contain": "behavioral",       "must_not": "algorithm",    "category": "semantic-hard"},
    {"query": "the expected return of an asset relative to market movement",      "must_contain": "beta",             "must_not": "bond",         "category": "semantic-hard"},
    # BM25-Hard
    {"query": "alpha generation strategies in quantitative funds",                "must_contain": "hedge fund",       "must_not": "capital asset","category": "bm25-hard"},
    {"query": "option pricing model Black Scholes assumptions",                   "must_contain": "option",           "must_not": "bond",         "category": "bm25-hard"},
    {"query": "market efficiency hypothesis and asset pricing anomalies",         "must_contain": "efficient",        "must_not": "portfolio",    "category": "bm25-hard"},
    {"query": "central bank interest rate impact on bond yields",                 "must_contain": "interest rate",    "must_not": "stock",        "category": "bm25-hard"},
    {"query": "currency pair volatility and forex trading risk",                  "must_contain": "foreign exchange", "must_not": "equity",       "category": "bm25-hard"},
    # Hybrid-Wins
    {"query": "LLM transformers predicting earnings from financial reports",      "must_contain": "language model",   "must_not": "portfolio",    "category": "hybrid-wins"},
    {"query": "NLP sentiment signals from news for stock returns",                "must_contain": "sentiment",        "must_not": "bond",         "category": "hybrid-wins"},
    {"query": "RAG retrieval augmented generation for financial question answering","must_contain": "retrieval",      "must_not": "derivative",   "category": "hybrid-wins"},
    {"query": "Sharpe ratio calculation for risk adjusted performance",           "must_contain": "sharpe",           "must_not": "behavioral",   "category": "hybrid-wins"},
    {"query": "deep learning models for high frequency algorithmic trading",      "must_contain": "algorithm",        "must_not": "fundamental",  "category": "hybrid-wins"},
]
print(f"✅ Eval set: {len(HARD_EVAL_SET)} hard queries across 3 categories")

✅ Eval set: 15 hard queries across 3 categories


In [13]:
def evaluate_hard(retriever, eval_set, alpha=0.5, top_k=5):
    retriever.alpha, retriever.top_k = alpha, top_k
    latencies, reciprocals, hits, neg_penalty = [], [], [], []
    cat_scores = {"semantic-hard": [], "bm25-hard": [], "hybrid-wins": []}
    details    = []

    for item in eval_set:
        start   = time.perf_counter()
        results = retriever.invoke(item["query"])
        elapsed = (time.perf_counter() - start) * 1000
        texts   = [doc.page_content.lower() for doc in results]

        rr, hit = 0.0, False
        for rank, text in enumerate(texts, 1):
            if item["must_contain"].lower() in text:
                rr, hit = 1.0 / rank, True
                break

        neg_hit = item["must_not"].lower() in texts[0] if texts else False
        latencies.append(elapsed); reciprocals.append(rr)
        hits.append(hit); neg_penalty.append(neg_hit)
        cat_scores[item["category"]].append(rr)
        details.append({"query": item["query"], "category": item["category"],
                        "hit": hit, "rr": round(rr, 3), "neg_penalty": neg_hit,
                        "latency_ms": round(elapsed, 1),
                        "top1": results[0].page_content[:100] if results else ""})

    return {
        "alpha": alpha,
        "mrr": round(statistics.mean(reciprocals), 4),
        "hit_rate": round(sum(hits) / len(hits) * 100, 1),
        "neg_rate": round(sum(neg_penalty) / len(neg_penalty) * 100, 1),
        "avg_latency": round(statistics.mean(latencies), 1),
        "p95_latency": round(sorted(latencies)[int(len(latencies) * 0.95)], 1),
        "cat_mrr": {k: round(statistics.mean(v), 4) if v else 0 for k, v in cat_scores.items()},
        "details": details,
    }

print("✅ Evaluation function ready.")

✅ Evaluation function ready.


In [14]:
mode_labels  = {0.0: "Pure BM25", 0.25: "BM25-leaning", 0.5: "Balanced Hybrid", 0.75: "Semantic-leaning", 1.0: "Pure Semantic"}
hard_results = []

print(f"{'Alpha':<8} {'Mode':<22} {'MRR@5':<10} {'Hit@5':<12} {'Neg Rate':<12} {'Avg ms'}")
print("-" * 78)
for alpha in [0.0, 0.25, 0.5, 0.75, 1.0]:
    r = evaluate_hard(retriever, HARD_EVAL_SET, alpha=alpha)
    hard_results.append(r)
    print(f"{alpha:<8} {mode_labels[alpha]:<22} {r['mrr']:<10} {r['hit_rate']}%{'':<8} {r['neg_rate']}%{'':<8} {r['avg_latency']} ms")
retriever.alpha = 0.5

Alpha    Mode                   MRR@5      Hit@5        Neg Rate     Avg ms
------------------------------------------------------------------------------
0.0      Pure BM25              0.6389     80.0%         0.0%         278.5 ms
0.25     BM25-leaning           0.6333     86.7%         0.0%         270.3 ms
0.5      Balanced Hybrid        0.73       86.7%         0.0%         269.6 ms
0.75     Semantic-leaning       0.6633     80.0%         0.0%         268.6 ms
1.0      Pure Semantic          0.6156     86.7%         0.0%         272.9 ms


In [15]:
# Per-category MRR breakdown
print(f"{'Alpha':<8} {'Semantic-Hard':<18} {'BM25-Hard':<16} {'Hybrid-Wins'}")
print("-" * 58)
for r in hard_results:
    print(f"{r['alpha']:<8} {r['cat_mrr']['semantic-hard']:<18} {r['cat_mrr']['bm25-hard']:<16} {r['cat_mrr']['hybrid-wins']}")

Alpha    Semantic-Hard      BM25-Hard        Hybrid-Wins
----------------------------------------------------------
0.0      0.45               0.6667           0.8
0.25     0.5                0.7              0.7
0.5      0.65               0.84             0.7
0.75     0.45               0.84             0.7
1.0      0.3667             0.84             0.64


In [16]:
# Detailed per-query breakdown for best config
best = max(hard_results, key=lambda x: x["mrr"])
print(f"\n📋 Best: alpha={best['alpha']} ({mode_labels[best['alpha']]}) | MRR: {best['mrr']} | Hit@5: {best['hit_rate']}% | Latency: {best['avg_latency']} ms\n")
print(f"{'#':<4} {'Category':<16} {'Hit':<6} {'RR':<7} {'Neg?':<7} {'Query'}")
print("-" * 90)
for i, d in enumerate(best["details"], 1):
    print(f"{i:<4} {d['category']:<16} {'✅' if d['hit'] else '❌':<6} {d['rr']:<7} {'⚠️' if d['neg_penalty'] else '✅':<7} {d['query'][:55]}")


📋 Best: alpha=0.5 (Balanced Hybrid) | MRR: 0.73 | Hit@5: 86.7% | Latency: 269.6 ms

#    Category         Hit    RR      Neg?    Query
------------------------------------------------------------------------------------------
1    semantic-hard    ✅      1.0     ✅       measuring how much money you could lose on a bad day
2    semantic-hard    ✅      0.25    ✅       spreading investments to reduce exposure to any single 
3    semantic-hard    ✅      1.0     ✅       using past price patterns to predict where a stock goes
4    semantic-hard    ✅      1.0     ✅       investors making irrational decisions based on emotions
5    semantic-hard    ❌      0.0     ✅       the expected return of an asset relative to market move
6    bm25-hard        ✅      0.2     ✅       alpha generation strategies in quantitative funds
7    bm25-hard        ✅      1.0     ✅       option pricing model Black Scholes assumptions
8    bm25-hard        ✅      1.0     ✅       market efficiency hypothesis and asset 

In [17]:
# Final summary + resume bullet
bm25_r     = next(r for r in hard_results if r["alpha"] == 0.0)
semantic_r = next(r for r in hard_results if r["alpha"] == 1.0)
hybrid_r   = next(r for r in hard_results if r["alpha"] == 0.5)
best_r     = max(hard_results, key=lambda x: x["mrr"])
gain_bm25  = round((hybrid_r['mrr'] - bm25_r['mrr'])     / max(bm25_r['mrr'], 0.001)     * 100, 1)
gain_sem   = round((hybrid_r['mrr'] - semantic_r['mrr']) / max(semantic_r['mrr'], 0.001) * 100, 1)

print("=" * 65)
print("   FINANCIAL HYBRID SEARCH — BENCHMARK SUMMARY")
print("=" * 65)
print(f"""
Corpus : {len(valid_chunks)} chunks | 20 Wikipedia + 10 ArXiv papers
Queries: {len(HARD_EVAL_SET)} hard queries across 3 difficulty categories

MRR@5:
  Pure BM25  (alpha=0.0) : {bm25_r['mrr']}
  Hybrid     (alpha=0.5) : {hybrid_r['mrr']}  WINNER
  Pure Dense (alpha=1.0) : {semantic_r['mrr']}

Hit Rate@5:
  BM25 / Hybrid / Dense  : {bm25_r['hit_rate']}% / {hybrid_r['hit_rate']}% / {semantic_r['hit_rate']}%

Latency: avg {hybrid_r['avg_latency']} ms | P95 {hybrid_r['p95_latency']} ms
Hybrid MRR gain vs BM25     : {gain_bm25:+}%
Hybrid MRR gain vs Semantic : {gain_sem:+}%
""")
print("=" * 65)
print("\n📌 RESUME BULLET:")
print("-" * 65)
print(f"""
Financial Knowledge Hybrid Search Engine
- Ingested {len(valid_chunks)}+ financial document chunks from 30 sources
  (Wikipedia + ArXiv) via automated API ingestion pipelines
- Combined BM25 sparse encoding with all-MiniLM-L6-v2 dense embeddings
  on Pinecone Serverless; benchmarked 5 alpha configs on 15 hard
  queries across 3 difficulty categories (semantic, keyword, hybrid-critical)
- Hybrid fusion (alpha=0.5) achieved MRR@5 of {hybrid_r['mrr']} and {hybrid_r['hit_rate']}%
  Hit Rate@5 at {hybrid_r['avg_latency']} ms avg latency, outperforming pure BM25
  by {gain_bm25:+}% and pure semantic by {gain_sem:+}% MRR
- Deployed as interactive Streamlit app with real-time alpha control
- Tech: Python, LangChain, Pinecone Serverless, HuggingFace, Streamlit
""")

   FINANCIAL HYBRID SEARCH — BENCHMARK SUMMARY

Corpus : 2195 chunks | 20 Wikipedia + 10 ArXiv papers
Queries: 15 hard queries across 3 difficulty categories

MRR@5:
  Pure BM25  (alpha=0.0) : 0.6389
  Hybrid     (alpha=0.5) : 0.73  WINNER
  Pure Dense (alpha=1.0) : 0.6156

Hit Rate@5:
  BM25 / Hybrid / Dense  : 80.0% / 86.7% / 86.7%

Latency: avg 269.6 ms | P95 280.2 ms
Hybrid MRR gain vs BM25     : +14.3%
Hybrid MRR gain vs Semantic : +18.6%


📌 RESUME BULLET:
-----------------------------------------------------------------

Financial Knowledge Hybrid Search Engine
- Ingested 2195+ financial document chunks from 30 sources
  (Wikipedia + ArXiv) via automated API ingestion pipelines
- Combined BM25 sparse encoding with all-MiniLM-L6-v2 dense embeddings
  on Pinecone Serverless; benchmarked 5 alpha configs on 15 hard
  queries across 3 difficulty categories (semantic, keyword, hybrid-critical)
- Hybrid fusion (alpha=0.5) achieved MRR@5 of 0.73 and 86.7%
  Hit Rate@5 at 269.6 ms avg la

## 12. Save Config for Streamlit App

In [18]:
config = {
    "index_name"   : INDEX_NAME,
    "bm25_path"    : "bm25_financial.json",
    "model_name"   : "all-MiniLM-L6-v2",
    "default_alpha": 0.5,
    "top_k"        : 5,
    "total_chunks" : len(valid_chunks),
}
with open("retriever_config.json", "w") as f:
    json.dump(config, f, indent=2)

print("✅ Config saved to retriever_config.json")
print("\nNext step: run  streamlit run app.py  in your terminal")
print(json.dumps(config, indent=2))

✅ Config saved to retriever_config.json

Next step: run  streamlit run app.py  in your terminal
{
  "index_name": "financial-hybrid-search",
  "bm25_path": "bm25_financial.json",
  "model_name": "all-MiniLM-L6-v2",
  "default_alpha": 0.5,
  "top_k": 5,
  "total_chunks": 2195
}
